# A Code Walkthrough of the Heap-Based Trading Engine

This notebook explains how `engine_heap.py` improves on the naive matching engine. The main optimization is replacing repeated full-list sorting with heaps, so the best buy and best sell are always cheap to access.

### 1. Why the Naive Engine Gets Slow

In `engine_naive.py`, every incoming order can trigger these expensive steps:

1. Filter the whole buy list.
2. Filter the whole sell list.
3. Sort the entire buy side.
4. Sort the entire sell side.

That means the engine keeps redoing work even though only a small part of the book usually matters: the best bid and the best ask.

### 2. Heap Optimization in One Idea

A heap keeps the most important order at the top:

- Buy heap: highest price first.
- Sell heap: lowest price first.

So instead of sorting the whole book on every order, we only push or pop around the top of the heap.

Typical improvement:

- Naive insertion plus reorder: often `O(n log n)` because of repeated sorts.
- Heap insertion: `O(log n)`.
- Best price lookup: `O(1)` from `heap[0]`.
- Remove top order after fill: `O(log n)`.

In [3]:
import sys
from pathlib import Path
from pprint import pprint

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in candidates:
    if (candidate / "engine_heap.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from engine_heap import Order, OrderBook

### 3. The Core Data Structures

`engine_heap.py` uses the same market concepts as the naive engine:

- `Order`: stores side, price, quantity, remaining quantity, and time.
- `Trade`: stores the execution record.
- `OrderBook`: stores orders, trades, and two heaps.

In [4]:
book = OrderBook()
print(type(book.buys), type(book.sells))

<class 'list'> <class 'list'>


### 4. How Python Turns These Lists Into Heaps

Python's `heapq` module expects the 'smallest' item to come first. For sells, that is exactly what we want. For buys, `engine_heap.py` flips the comparison in `Order.__lt__` so a higher buy price behaves like the smallest heap element.

That trick gives us:

- best buy at `buys[0]`
- best sell at `sells[0]`

without sorting the whole list every time.

In [ ]:
buy_a = Order("buy", "100.00", 10)
buy_b = Order("buy", "101.00", 10)
sell_a = Order("sell", "102.00", 10)
sell_b = Order("sell", "101.50", 10)

print("Higher buy should win:", buy_b < buy_a)
print("Lower sell should win:", sell_b < sell_a)

### 5. Matching Flow

When a buy order arrives:

1. Look at the best sell only.
2. If prices cross, execute against that top sell.
3. If the resting sell is fully filled, pop it from the heap.
4. Continue until the new order is filled or the prices no longer cross.

The sell path is symmetric.

In [ ]:
book = OrderBook()

book.place_order(Order("buy", "100.00", 10))
book.place_order(Order("buy", "101.00", 5))
book.place_order(Order("sell", "103.00", 7))

print("Order book depth before crossing order:")
pprint(book.get_order_book_depth())

book.place_order(Order("sell", "100.50", 6))

print("\nOrder book depth after matching:")
pprint(book.get_order_book_depth())

print("\nTrades:")
pprint(list(book.trades.values()))

### 6. What Became Faster

Using heaps improves the hot path of the engine:

- We no longer sort all buys and sells on each order.
- We always inspect the best available price first.
- Fully filled top orders are removed with a `heapq.heappop`, not a full reorder.
- Insertion and removal scale better as the book grows.

### 7. A Small Side-by-Side Complexity View

| Operation | Naive List Book | Heap Book |
| --- | --- | --- |
| Add resting order | append + later full sort | `heappush` |
| Best bid / ask lookup | after sort | direct from heap root |
| Remove filled best order | `pop(0)` after sort | `heappop` |
| Reordering cost | repeated `O(n log n)` work | `O(log n)` local fix-up |

### 8. The Trade-Offs

Heaps are faster for best-price access, but they are not perfect for everything:

- Cancelling an arbitrary order in the middle of a heap is awkward.
- This engine uses lazy cleanup: mark an order filled/cancelled, then discard it when it reaches the top.
- Getting a fully sorted market-depth view still requires iterating through the heap contents.

So the heap design optimizes the matching path first, which is usually the most important path in a trading engine.

In [ ]:
print("Best buy at heap root:", book.buys[0] if book.buys else None)
print("Best sell at heap root:", book.sells[0] if book.sells else None)
print("Last traded price:", book.last_trading_price)

### 9. Summary

The heap-based engine is faster because it stops re-sorting the whole book. Instead, it maintains just enough structure so the next match is always cheap to find. That is the main optimization: pay a small `log n` maintenance cost on updates so you avoid large repeated sorting costs on every order.